In [36]:
# ---------------------------
# 📌 1️⃣ Imports & Setup
# ---------------------------
import os
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langchain_community.llms import Ollama
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

from typing import TypedDict
import json

# Load your .env with SERPER_API_KEY
load_dotenv()

# Ollama must be running locally
ollama_llm = Ollama(
    base_url="http://localhost:11434",
    model="llama3.2"  # or your local model
)

search = TavilySearchAPIWrapper(
    tavily_api_key=os.environ["TAVILY_API_KEY"]
)

In [37]:
search.results("Capital of India", max_results=2)

[{'title': 'New Delhi - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/New_Delhi',
  'content': 'Appearance\n\nmove to sidebar hide\n\nCoordinates: 28°36′50″N 77°12′32″E / 28.61389°N 77.20889°E / 28.61389; 77.20889\n\nImage 4: Page semi-protected\n\nFrom Wikipedia, the free encyclopedia\n\nCapital city of India\n\nThis article is about the capital of India, within the union territory of Delhi. For other uses, see New Delhi (disambiguation) "New Delhi (disambiguation)"). [...] New Delhi (/ˈ nj uː ˈ d ɛ.l i/ⓘ;( _\\_\\\\_Naī Dillī\\\\_\\__, pronounced( "Help:IPA/Hindi and Urdu")) is the capital of India and a part of the National Capital Territory of Delhi (NCT). New Delhi is the seat of all three branches of the Government of India, hosting the Rashtrapati Bhavan, Sansad Bhavan, and the Supreme Court. New Delhi is a municipality within the NCT, administered by the New Delhi Municipal Council (NDMC), which covers mostly Lutyens\' Delhi and a few adjacent areas. The municipal',
  'sco

In [38]:
# ---------------------------
# 📌 2️⃣ Shared State
# ---------------------------
class ArticleState(TypedDict):
    topic: str
    draft: str
    feedback: str
    good_enough: bool
    final_article: str
    critic_count: int

In [43]:
# ---------------------------
# 📌 3️⃣ Researcher Node
# ---------------------------
def researcher(state: ArticleState):
    topic = state["topic"]
    feedback = state.get("feedback", "")

    # 🔍 Run Google Search
    search_results = search.results(topic, max_results=3)

    # 🧩 Build prompt: include search results + any feedback
    if feedback:
        refinement = f"\nAlso improve the draft using this feedback: {feedback}"
    else:
        refinement = ""

    prompt = f"""
    Topic: "{topic}"

    Below are real search results about this topic:
    {search_results}

    Using these search results, write or improve a detailed, beginner-friendly article. Feel free to add 
    some additional knowledge related to the topic that may not be in the search results.
    Be clear, factual, and helpful.
    {refinement}
    """

    print("\n🧩 [Researcher] Prompt sent to LLM:")
    # print(prompt)

    response = ollama_llm.invoke(prompt)

    print("\n✅ [Researcher] Draft produced.")
    return {"draft": response}


In [48]:

# ---------------------------
# 📌 4️⃣ Critic Node
# ---------------------------
def critic(state: ArticleState):
    draft = state["draft"]
    critic_count = state['critic_count'] + 1

    prompt = f"""
    Critique the following draft for clarity, accuracy, and beginner-friendliness:

    {draft}

    - If the draft is good enough to publish, return: {{"good_enough": true, "feedback": "Looks good!"}}
    - If the draft needs improvements, return: {{"good_enough": false, "feedback": "...specific feedback..."}}

    Respond ONLY in valid JSON.
    """

    print("\n🧐 [Critic] Prompt sent to LLM:")
    # print(prompt)

    response = ollama_llm.invoke(prompt)
    print(f"\n🧐 [Critic] Raw LLM output:\n{response}")

    # Parse JSON output
    try:
        data = json.loads(response)
        return {
            "good_enough": data["good_enough"],
            "feedback": data["feedback"],
            "critic_count": critic_count
        }
    except Exception as e:
        print("❌ Error parsing Critic output, assuming not good enough:", e)
        return {
            "good_enough": False,
            "feedback": "LLM response could not be parsed. Please clarify and improve.",
            "critic_count": critic_count
        }

In [49]:
# ---------------------------
# 📌 5️⃣ Formatter Node
# ---------------------------
def formatter(state: ArticleState):
    draft = state["draft"]
    feedback = state.get("feedback", "")

    prompt = f"""
    Here is the final draft:
    {draft}

    Here is the feedback:
    {feedback}

    Please polish this draft to be clear, engaging, and ready for publishing on Medium.
    Keep the tone friendly and beginner-friendly.
    """

    print("\n✨ [Formatter] Prompt sent to LLM:")
    # print(prompt)

    response = ollama_llm.invoke(prompt)

    print("\n✅ [Formatter] Final article produced.")
    return {"final_article": response}

In [50]:

# ---------------------------
# 📌 6️⃣ Build LangGraph
# ---------------------------
graph = StateGraph(ArticleState)

# Add nodes
graph.add_node("researcher", researcher)
graph.add_node("critic", critic)
graph.add_node("formatter", formatter)

# Entry point
graph.set_entry_point("researcher")

# Researcher → Critic
graph.add_edge("researcher", "critic")

# Critic conditional: Good → Formatter, Else → loop back to Researcher
def critic_router(state: ArticleState):
    print("\n🔄 [Critic Router] Evaluating state:", state['critic_count'])
    if state["good_enough"] or state["critic_count"] >= 1:
        return "formatter"
    else:
        return "researcher"

graph.add_conditional_edges(
    "critic",
    critic_router,
    {
        "formatter": "formatter",
        "researcher": "researcher"
    }
)

# Formatter → END
graph.add_edge("formatter", END)

# Compile
app = graph.compile()

# ---------------------------
# 📌 7️⃣ Run it!
# ---------------------------
result = app.invoke({
    "topic": "How to bake a simple cake at home",
    "critic_count": 0
})

print("\n🎉 ✅ ✅ ✅ FINAL ARTICLE:")
print(result["final_article"])



🧩 [Researcher] Prompt sent to LLM:

✅ [Researcher] Draft produced.

🧐 [Critic] Prompt sent to LLM:

🧐 [Critic] Raw LLM output:
{"good_enough": false, "feedback": "The draft could benefit from more detailed explanations of mixing techniques and a clearer distinction between the simple white cake recipe and the chocolate cake variation. Additionally, some of the tips for achieving perfect results could be rephrased or expanded upon to provide more actionable advice. Consider adding more visuals, such as images or diagrams, to illustrate key concepts and make the guide more engaging and accessible to beginners."}

🔄 [Critic Router] Evaluating state: 1

✨ [Formatter] Prompt sent to LLM:

✅ [Formatter] Final article produced.

🎉 ✅ ✅ ✅ FINAL ARTICLE:
Based on the feedback provided, I've revised the draft to address the suggested improvements. Here's a polished version of the guide:

**How to Bake a Simple Cake at Home: A Beginner's Guide**

Welcome to the world of cake baking! With this beg